In [ ]:
import os

os.environ['UNSLOTH_USE_MODELSCOPE'] = '1'  # unsloth import

os.environ["LD_LIBRARY_PATH"] = (
    "/usr/local/lib/python3.12/dist-packages/nvidia/cu13/lib:"
    + os.environ.get("LD_LIBRARY_PATH", "")
)

print("ModelScope:", os.environ['UNSLOTH_USE_MODELSCOPE'])
print("LD set ✓")

In [ ]:
!pip install modelscope -q

In [ ]:
# Install unsloth
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install trl==0.14.0 peft==0.14.0 -q
print("✅ Unsloth installed!")

In [ ]:
 #Step3: Import necessary libraries
import unsloth
from unsloth import FastLanguageModel, is_bfloat16_supported
import torch
from trl import SFTTrainer
from huggingface_hub import login
from transformers import TrainingArguments
from datasets import load_dataset
import wandb
print("✅ Imports done")

In [ ]:
from google.colab import userdata
from huggingface_hub import login
hf_token = userdata.get('HF_TOKEN')
login(hf_token)

In [ ]:
# Checking GPU availability
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU device:", torch.cuda.get_device_name(0)) if torch.cuda.is_available() else "No GPU"


In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/DeepSeek-R1-Distill-Llama-8B",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
    token = hf_token,
)
print("✅ Model loaded!")

In [ ]:
# Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:

import shutil

tokenizer.save_pretrained("/content/drive/MyDrive/AI-Doctor-Model")

src = "/root/.cache/modelscope/hub/models/unsloth/deepseek-r1-distill-llama-8b-unsloth-bnb-4bit"
dst = "/content/drive/MyDrive/AI-Doctor-Model"

print("Copying model files... ⏳")
shutil.copytree(src, dst, dirs_exist_ok=True)
print("✅ Model Drive loaded!")

In [ ]:
# Now I have to run this cell!
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "/content/drive/MyDrive/AI-Doctor-Model",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)
print("✅ Loaded from Drive - no download!")

In [ ]:
#Setup system prompt
prompt_style = """
Below is a task description along with additional context provided in the input section. Your goal is to provide a well-reasoned response that effectively addresses the request.

Below crafting your answer, take a moment to carefully analyze the question. Develop a clear, step-by-step thought process to ensure your response in both and accurate.

### Instruction:
You are a medical expert specializing in clinical reasoning, diagnostics, and treatment planning. Answer the medical question below using your advanced knowledge.

### Question:
{}

### Response:
{}
"""

In [ ]:
# Run Inference on the model

# Define a test question
question = """ A 61-year-old woman with a long history of involuntary urine loss duing activities like coughing or sneezing but no leakage at night undergoes a gynecological eaxm and Q-tip test. Based on these findings,what would cystometry most likely reveal about her residual volume and detrusor contractions.
"""

# Tokenize the input
inputs = tokenizer([prompt_style.format(question, "")], return_tensors="pt").to("cuda")

# Generate a response
outputs = model.generate (
    input_ids = inputs.input_ids,
    attention_mask = inputs.attention_mask,
    max_new_tokens = 1200, # max_new_token to max_new_tokens
    use_cache = True
)

# response tokens back to text
response = tokenizer.batch_decode(outputs, skip_special_tokens=True)

clean = response[0]

clean = clean.replace("Ġ", " ")
clean = clean.replace("Ċ", "\n")

# the answer after <think>...</think>
if "</think>" in clean:
    clean = clean.split("</think>")[-1].strip()
elif "### Response:" in clean:
    clean = clean.split("### Response:")[-1].strip()

print(clean)


In [ ]:
# setup fine-tuning

# Load Dataset
medical_dataset = load_dataset("FreedomIntelligence/medical-o1-reasoning-SFT","en",split = "train[:500]", trust_remote_code = True)

In [ ]:
medical_dataset[1]

In [ ]:
EOS_TOKEN = tokenizer.eos_token
EOS_TOKEN
# Define EOS_TOKEN (End Of Sentence) which tells the model when to stop generating text during training

In [ ]:
# Update the prompt_style for training

train_prompt_style = """
Below is a task description along with additional context provided in the input section. Your goal is to provide a well-reasoned response that effectively addresses the request.

Below crafting your answer, take a moment to carefully analyze the question. Develop a clear, step-by-step thought process to ensure your response in both and accurate.

### Instruction:
You are a medical expert specializing in clinical reasoning, diagnostics, and treatment planning. Answer the medical question below using your advanced knowledge.

### Question:
{}

### Response:
<think>
{}
</think>
{}
"""

In [ ]:

# Prepare the data for fine-tuning

def preprocess_input_data(examples):
  inputs = examples["Question"]
  cots = examples["Complex_CoT"]
  outputs = examples["Response"]

  texts = []
  for input, cot, output in zip(inputs, cots, outputs):
    text = train_prompt_style.format(input, cot, output) + EOS_TOKEN
    texts.append(text)

  return {
      "texts" : texts,
  }

In [ ]:
finetune_dataset = medical_dataset.map(
    preprocess_input_data,
    batched=True
)

In [ ]:
finetune_dataset["texts"][1]

In [ ]:
# Setup/Apply LoRA finetuning to the model
from peft import LoraConfig, get_peft_model

peft_config = LoraConfig(
    r = 16,
    lora_alpha = 16,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout = 0,
    bias = "none",
    task_type = "CAUSAL_LM",
)

model_lora = get_peft_model(model, peft_config)
model_lora.print_trainable_parameters()
print("LoRA ready!")

In [ ]:

if hasattr(model, '_unwrapped_old_generate'):
  del model._unwrapped_old_generate

In [ ]:
print(type(finetune_dataset["texts"][0]))
print(finetune_dataset["texts"][0][:500])

In [ ]:
print(finetune_dataset["texts"][0][-1000:])

In [ ]:
!pip install trl -q
print(" TRL installed!")

In [ ]:
max_sequence_length = 2048

from trl import SFTTrainer, SFTConfig

# Dataset
finetune_dataset_fixed = finetune_dataset.map(
    lambda x: {"text": x["texts"][0] if isinstance(x["texts"], list) else x["texts"]},
    remove_columns=["texts"]
)

trainer = SFTTrainer(
    model = model_lora,
    tokenizer = tokenizer,
    train_dataset = finetune_dataset_fixed,
    dataset_text_field = "text",
    args = SFTConfig(
        max_seq_length = max_sequence_length,
        dataset_num_proc = 1,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 1,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        gradient_checkpointing = False,
        save_strategy = "no",
        report_to = "none",
    )
)
print("Trainer ready!")

In [ ]:
# Setup, WANDB
from google.colab import userdata
wnb_token = userdata.get("WANDB_API_TOKEN")
#login to WnB
wandb.login(key=wnb_token) #import wandb
run = wandb.init(
    project='Fine-tune-DeepSeek-R1-on-Medical-CoT-Dataset',
    job_type="training",
    anonymous="allow"
)


In [ ]:

import os

model_lora.save_pretrained("outputs/final_model")
tokenizer.save_pretrained("outputs/final_model")
print("✅ Model saved successfully!")

In [ ]:
wandb.finish()

In [ ]:
# Testing after finetuning
question = """ A 61-year-old woman with a long history of involuntary urine loss duing activities like coughing or sneezing but no leakage at night undergoes a gynecological eaxm and Q-tip test. Based on these findings,what would cystometry most likely reveal about her residual volume and detrusor contractions.
"""

from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model_lora)

# Tokenize the input
inputs = tokenizer([prompt_style.format(question, "")], return_tensors="pt").to("cuda")

# Generate a response
outputs = model_lora.generate (
    input_ids = inputs.input_ids,
    attention_mask = inputs.attention_mask,
    max_new_tokens = 1200,
    use_cache = True
)


response = tokenizer.decode(outputs[0], skip_special_tokens=True)
response = response.replace("Ġ", " ").replace("Ċ", "\n")
if "</think>" in response:
    response = response.split("</think>")[-1].strip()

print(response)